# 02 · Train final model (Random Forest Tuned)

Este notebook entrena y exporta el **modelo final oficial del proyecto**:
**Random Forest Tuned**.

## Objetivos
- cargar los mejores hiperparámetros del notebook 01
- entrenar el modelo final de Random Forest
- evaluar su rendimiento en validación
- generar predicciones y matriz de confusión
- guardar el modelo y los outputs oficiales para Streamlit

## Resultado esperado
Al terminar este notebook tendrás:
- modelo `.joblib` final
- métricas finales
- predicciones de validación
- matriz de confusión en porcentaje
- modelo comprimido para despliegue


## 1. Imports y configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "churned"

EXPORT_DIR = Path("../data/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("../data/processed/train_model_ready.csv")

BEST_PARAMS_PATH = EXPORT_DIR / "tuned_model_best_params.csv"

print(f"Usando dataset: {DATA_PATH}")
print(f"Buscando mejores parámetros en: {BEST_PARAMS_PATH}")


Usando dataset: ../data/processed/train_model_ready.csv
Buscando mejores parámetros en: ../data/exports/tuned_model_best_params.csv


## 2. Carga de datos

In [22]:
df = pd.read_csv(DATA_PATH).copy()

print("Shape:", df.shape)
df.head()


Shape: (125000, 29)


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,...,churned,tenure_days,tenure_months,songs_per_hour,high_skip_user,age_group,weekly_hours_bin,skip_rate_bin,state_code,region
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,...,0,1606,53.533333,7.547554,0,25-34,20-30,0-0.2,MT,West
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,...,1,2897,96.566667,1.877504,1,50-64,20-30,0.8-1.0,NJ,Northeast
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,...,0,348,11.600000,15.843835,0,50-64,10-20,0-0.2,WA,West
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,...,0,2894,96.466667,19.350248,0,50-64,20-30,0-0.2,CA,West
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,...,0,92,3.066667,10.496233,0,50-64,20-30,0-0.2,WA,West


## 3. Preparación del dataset

Eliminamos columnas auxiliares si existen y separamos predictores y objetivo.


In [23]:
drop_cols = [col for col in ["y_true", "y_pred", "p_churn"] if col in df.columns]

model_df = df.drop(columns=drop_cols).copy()

if TARGET not in model_df.columns:
    raise ValueError(f"No se encontró la columna objetivo '{TARGET}'.")

X = model_df.drop(columns=[TARGET]).copy()
y = model_df[TARGET].copy()

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))
print("Shape X:", X.shape)
print("Shape y:", y.shape)


Variables numéricas: 18
Variables categóricas: 10
Shape X: (125000, 28)
Shape y: (125000,)


## 4. Train / validation split

Mantenemos un split reproducible y estratificado para:
- entrenar el modelo final
- generar outputs consistentes para la app

En este caso hemos realizado el split 90/10 para train/test debido al gran número de casos que tenemos.


In [24]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.1,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)


Train shape: (112500, 28)
Valid shape: (12500, 28)


## 5. Preprocesador

Para Random Forest:
- imputación en numéricas
- imputación + one-hot en categóricas
- sin escalado


In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 6. Cargar los mejores hiperparámetros del notebook 01

Si el archivo `tuned_model_best_params.csv` existe, usamos la fila de **RF Tuned**.  
Si no existe, usamos una configuración fallback razonable.


In [26]:
if BEST_PARAMS_PATH.exists():
    best_params_df = pd.read_csv(BEST_PARAMS_PATH)
    rf_params_row = best_params_df[best_params_df["model"] == "RF Tuned"].copy()
else:
    best_params_df = None
    rf_params_row = pd.DataFrame()

rf_params_row


,model,classifier__solver,classifier__class_weight,classifier__C,classifier__n_estimators,classifier__min_samples_split,classifier__min_samples_leaf,classifier__max_features,classifier__max_depth,classifier__model__learning_rate,classifier__model__hidden_units,classifier__model__dropout_rate,classifier__epochs,classifier__batch_size
1,RF Tuned,NaN,NaN,NaN,300.0,10.0,2.0,NaN,12.0,NaN,NaN,NaN,NaN,NaN


In [27]:
def coerce_none(value):
    if pd.isna(value):
        return None
    if isinstance(value, str) and value.strip().lower() == "none":
        return None
    return value

if len(rf_params_row) > 0:
    rf_best_params = {
        "n_estimators": int(rf_params_row["classifier__n_estimators"].iloc[0]),
        "max_depth": coerce_none(rf_params_row["classifier__max_depth"].iloc[0]),
        "min_samples_split": int(rf_params_row["classifier__min_samples_split"].iloc[0]),
        "min_samples_leaf": int(rf_params_row["classifier__min_samples_leaf"].iloc[0]),
        "max_features": coerce_none(rf_params_row["classifier__max_features"].iloc[0]),
        "class_weight": coerce_none(rf_params_row["classifier__class_weight"].iloc[0]),
    }
else:
    rf_best_params = {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": None,
    }

rf_best_params


{'n_estimators': 300,
 'max_depth': 12.0,
 'min_samples_split': 10,
 'min_samples_leaf': 2,
 'max_features': None,
 'class_weight': None}

**Nota**: el parámetro max_depth no se acepta como `.float_64`, por lo que se convierte a entero antes de la construcción del pipeline

In [28]:
rf_best_params["max_depth"] = int(rf_best_params["max_depth"]) if rf_best_params["max_depth"] is not None else None

## 7. Construcción del pipeline final

In [29]:
rf_final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=rf_best_params["n_estimators"],
        max_depth=rf_best_params["max_depth"],
        min_samples_split=rf_best_params["min_samples_split"],
        min_samples_leaf=rf_best_params["min_samples_leaf"],
        max_features=rf_best_params["max_features"],
        class_weight=rf_best_params["class_weight"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_final_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [30]:
print(rf_best_params)
for k, v in rf_best_params.items():
    print(k, v, type(v))

{'n_estimators': 300, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'class_weight': None}
n_estimators 300 <class 'int'>
max_depth 12 <class 'int'>
min_samples_split 10 <class 'int'>
min_samples_leaf 2 <class 'int'>
max_features None <class 'NoneType'>
class_weight None <class 'NoneType'>


## 8. Entrenamiento del modelo final

In [31]:
rf_final_model.fit(X_train, y_train)
print("Modelo final entrenado correctamente.")


Modelo final entrenado correctamente.


## 9. Predicciones y métricas finales

In [32]:
y_pred = rf_final_model.predict(X_valid)
y_proba = rf_final_model.predict_proba(X_valid)[:, 1]

final_metrics = {
    "model": "RF Tuned",
    "accuracy": accuracy_score(y_valid, y_pred),
    "precision": precision_score(y_valid, y_pred, zero_division=0),
    "recall": recall_score(y_valid, y_pred, zero_division=0),
    "f1": f1_score(y_valid, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, y_proba),
}

final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df


,model,accuracy,precision,recall,f1,roc_auc
0,RF Tuned,0.84992,0.851307,0.85741,0.854348,0.939612


## 10. Matriz de confusión

In [34]:
cm = confusion_matrix(y_valid, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Pred No Churn", "Pred Churn"]
)

cm_df


,Pred No Churn,Pred Churn
Actual No Churn,5122,961
Actual Churn,915,5502


In [35]:
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

cm_pct_df = pd.DataFrame(
    cm_pct,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Pred No Churn", "Pred Churn"]
)

cm_pct_df


,Pred No Churn,Pred Churn
Actual No Churn,84.201874,15.798126
Actual Churn,14.259000,85.741000


In [36]:
fig_cm = px.imshow(
    cm_pct_df,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Reds",
    title="Matriz de confusión normalizada (%) - RF Tuned"
)

fig_cm.update_traces(texttemplate="%{z:.2f}%")
fig_cm.update_layout(
    title_x=0.5,
    xaxis_title="Clase predicha",
    yaxis_title="Clase real"
)

fig_cm.show()


## 11. Predicciones de validación

In [37]:
preds_df = X_valid.copy()
preds_df["y_true"] = y_valid.values
preds_df["y_pred"] = y_pred
preds_df["p_churn"] = y_proba

preds_df = preds_df.sort_values("p_churn", ascending=False).reset_index(drop=True)
preds_df.head(20)


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,...,songs_per_hour,high_skip_user,age_group,weekly_hours_bin,skip_rate_bin,state_code,region,y_true,y_pred,p_churn
0,99147,78,North Carolina,Free,Monthly,2,Debit Card,High,-510,12.582860,...,9.695729,1,65-79,10-20,0.6-0.8,NC,South,1,1,1.0
1,9006,72,New Jersey,Free,Yearly,1,Debit Card,High,-2590,21.356337,...,11.940250,0,65-79,20-30,0-0.2,NJ,Northeast,1,1,1.0
2,79393,66,West Virginia,Free,Yearly,3,Credit Card,High,-714,8.829411,...,53.344439,1,65-79,5-10,0.8-1.0,WV,South,1,1,1.0
3,63230,70,North Carolina,Free,Monthly,3,Paypal,High,-1563,33.358183,...,10.732000,0,65-79,30-40,0.4-0.6,NC,South,1,1,1.0
4,60602,18,Vermont,Free,Yearly,3,Paypal,High,-1841,14.425132,...,20.034478,0,18-24,10-20,0.4-0.6,VT,Northeast,1,1,1.0
5,43471,70,North Carolina,Free,Yearly,3,Paypal,High,-1538,19.497724,...,8.565102,1,65-79,10-20,0.6-0.8,NC,South,1,1,1.0
6,59921,71,Alabama,Free,Monthly,3,Credit Card,High,-1811,30.921954,...,15.943365,1,65-79,30-40,0.8-1.0,AL,South,1,1,1.0
7,43384,22,Montana,Free,Yearly,2,Apple Pay,High,-2056,9.272060,...,38.071369,1,18-24,5-10,0.8-1.0,MT,West,1,1,1.0
8,743,75,Nebraska,Free,Monthly,4,Paypal,High,-2455,4.692827,...,2.557094,0,65-79,0-5,0.4-0.6,NE,Midwest,1,1,1.0
9,80340,69,Idaho,Free,Monthly,3,Debit Card,High,-378,31.174591,...,4.394605,1,65-79,30-40,0.6-0.8,ID,West,1,1,1.0


## 12. Clientes activos con mayor riesgo

Útil para negocio: clientes que todavía no han churnado pero tienen alta probabilidad estimada.


In [38]:
high_risk_active = preds_df[preds_df["y_true"] == 0].copy()
high_risk_active.head(20)


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,...,songs_per_hour,high_skip_user,age_group,weekly_hours_bin,skip_rate_bin,state_code,region,y_true,y_pred,p_churn
1120,81171,45,Alabama,Premium,Monthly,4,Credit Card,Low,-1428,2.287537,...,27.540542,1,35-49,0-5,0.8-1.0,AL,South,0,1,0.996078
1186,13192,44,South Carolina,Family,Monthly,2,Apple Pay,High,-881,7.477287,...,9.896638,1,35-49,5-10,0.6-0.8,SC,South,0,1,0.995586
1488,87457,53,Maine,Student,Monthly,4,Debit Card,Medium,-612,33.160606,...,10.856255,1,50-64,30-40,0.8-1.0,ME,Northeast,0,1,0.991829
1648,63484,58,Virginia,Family,Monthly,4,Apple Pay,Medium,-2348,1.562460,...,278.407073,0,50-64,0-5,0.4-0.6,VA,South,0,1,0.989709
1673,78253,43,Utah,Free,Yearly,2,Credit Card,High,-1721,10.017442,...,23.558907,0,35-49,10-20,0.6-0.8,UT,West,0,1,0.989389
1706,114547,35,Washington,Family,Monthly,3,Credit Card,High,-764,37.697901,...,8.753803,1,25-34,30-40,0.8-1.0,WA,West,0,1,0.988910
1718,15762,53,Utah,Free,Yearly,4,Apple Pay,Low,-1973,11.414083,...,38.286038,1,50-64,10-20,0.8-1.0,UT,West,0,1,0.988680
1733,35485,54,Utah,Premium,Yearly,4,Debit Card,High,-1943,11.737068,...,6.304811,1,50-64,10-20,0.6-0.8,UT,West,0,1,0.988507
1778,86814,52,Vermont,Free,Monthly,3,Credit Card,Medium,-357,33.362847,...,13.218296,0,50-64,30-40,0.2-0.4,VT,Northeast,0,1,0.987761
1787,24253,51,New Jersey,Free,Yearly,1,Debit Card,Medium,-777,22.085414,...,19.877372,1,50-64,20-30,0.6-0.8,NJ,Northeast,0,1,0.987576


## 13. Importancia de variables

In [39]:
feature_names = rf_final_model.named_steps["preprocessor"].get_feature_names_out()
feature_importances = rf_final_model.named_steps["classifier"].feature_importances_

rf_feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

rf_feature_importance_df.head(20)


,feature,importance
0,num__weekly_hours,0.210586
1,cat__subscription_type_Free,0.151766
2,cat__customer_service_inquiries_High,0.137993
3,num__age,0.085965
4,num__song_skip_rate,0.079060
5,num__num_subscription_pauses,0.076661
6,cat__subscription_type_Student,0.058735
7,cat__customer_service_inquiries_Low,0.043811
8,cat__customer_service_inquiries_Medium,0.024137
9,num__notifications_clicked,0.020593


## 14. Guardado del modelo

In [40]:
rf_model_path = MODEL_DIR / "rf_tuned_model.joblib"
rf_model_compressed_path = MODEL_DIR / "rf_tuned_model_compressed.joblib"

joblib.dump(rf_final_model, rf_model_path)
joblib.dump(rf_final_model, rf_model_compressed_path, compress=3)

print("Modelo guardado en:")
print("-", rf_model_path)
print("-", rf_model_compressed_path)

print("\nTamaños:")
print("Original:", round(rf_model_path.stat().st_size / (1024**2), 2), "MB")
print("Comprimido:", round(rf_model_compressed_path.stat().st_size / (1024**2), 2), "MB")


Modelo guardado en:
- ../models/rf_tuned_model.joblib
- ../models/rf_tuned_model_compressed.joblib

Tamaños:
Original: 65.89 MB
Comprimido: 19.62 MB


## 15. Exportación oficial para la app

In [41]:
final_metrics_df.to_csv(EXPORT_DIR / "rf_tuned_metrics.csv", index=False)
preds_df.to_csv(EXPORT_DIR / "rf_tuned_validation_predictions.csv", index=False)
cm_pct_df.to_csv(EXPORT_DIR / "rf_tuned_confusion_matrix_percentage.csv")
rf_feature_importance_df.to_csv(EXPORT_DIR / "rf_tuned_feature_importance.csv", index=False)

print("Archivos exportados:")
print("-", EXPORT_DIR / "rf_tuned_metrics.csv")
print("-", EXPORT_DIR / "rf_tuned_validation_predictions.csv")
print("-", EXPORT_DIR / "rf_tuned_confusion_matrix_percentage.csv")
print("-", EXPORT_DIR / "rf_tuned_feature_importance.csv")


Archivos exportados:
- ../data/exports/rf_tuned_metrics.csv
- ../data/exports/rf_tuned_validation_predictions.csv
- ../data/exports/rf_tuned_confusion_matrix_percentage.csv
- ../data/exports/rf_tuned_feature_importance.csv


## 16. Conclusión

Con este notebook ya tienes el **Random Forest Tuned** listo para producción dentro del proyecto:

- entrenado con los mejores hiperparámetros disponibles
- evaluado en validación
- exportado para Streamlit
- guardado como artefacto `.joblib`
